# 03 — Train dan Evaluasi Model NER

**Jalankan notebook ini hanya setelah ketiga anggota selesai mengisi file anotasi dan melakukan pemeriksaan silang.** Notebook ini akan:

1. membaca file anotasi ketiga anggota;
2. memvalidasi label dan offset karakter;
3. membagi data 70/15/15 berdasarkan ID artikel;
4. mengukur baseline model pretrained;
5. melakukan fine-tuning/adaptasi NER pada anotasi train;
6. memilih checkpoint dari hasil validation;
7. mengukur test set satu kali di akhir, termasuk exact-span precision, recall, F1, metrik per label, dan analisis kesalahan.

Hasil metrik tidak ditentukan sebelumnya; hasil aktual bergantung pada anotasi, pembagian data, dan model.

## Pembagian peran

- **Anggota 1:** kumpulkan tiga CSV anotasi, pastikan ID artikel dan teks benar, periksa konflik/offset.
- **Anggota 2:** jalankan baseline, training, validation, dan evaluasi; catat parameter serta hasil.
- **Anggota 3:** setelah model diekspor, integrasikan artefak model ke API/web dan buat tes integrasi.

Semua anggota harus membantu mengecek anotasi. Test set tidak boleh digunakan untuk memperbaiki anotasi, preprocessing, atau memilih hyperparameter setelah hasil test dilihat.

In [ ]:
import io, json, zipfile, random, shutil, os
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display, Markdown
from google.colab import files

SEED = 42
MAX_EPOCHS = 20
PATIENCE = 4
OUT = Path('/content/ner_training_output')
OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
print('Lingkungan training siap. Seed =', SEED)

## 1. Unggah hasil anotasi dari ketiga anggota

Unggah tiga file `annotation_queue_member_1.csv`, `annotation_queue_member_2.csv`, dan `annotation_queue_member_3.csv` yang sudah diisi. Boleh juga mengunggah satu ZIP berisi ketiga file CSV.

Pastikan kolom `annotations_json` sudah diisi dengan JSON yang valid; jika artikel tidak memiliki entitas, isi `[]`, bukan sel kosong.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('Unggah tiga CSV anotasi atau ZIP yang berisi ketiganya.')

annotation_frames = []
def read_csv_blob(blob, source_name):
    last_error = None
    for enc in ('utf-8-sig','utf-8','latin-1'):
        try:
            df = pd.read_csv(io.BytesIO(blob), dtype=str, encoding=enc, keep_default_na=False)
            df.columns = [str(c).strip().lower() for c in df.columns]
            df['annotation_source_file'] = source_name
            return df
        except Exception as e:
            last_error = e
    raise RuntimeError(f'Gagal membaca {source_name}: {last_error}')

for name, blob in uploaded.items():
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            for member in zf.namelist():
                if member.lower().endswith('.csv') and ('annotation_queue_member_' in member.lower() or 'annotation' in member.lower()):
                    annotation_frames.append(read_csv_blob(zf.read(member), member))
    elif name.lower().endswith('.csv'):
        annotation_frames.append(read_csv_blob(blob, name))

if not annotation_frames:
    raise ValueError('Tidak menemukan CSV anotasi yang dapat dibaca.')
ann_df = pd.concat(annotation_frames, ignore_index=True, sort=False)
print('File/sumber CSV dibaca:', len(annotation_frames))
print('Jumlah baris anotasi:', len(ann_df))
display(ann_df.head(3))

## 2. Validasi format anotasi

Notebook menolak sel anotasi kosong atau offset yang tidak cocok. `start` termasuk karakter awal dan `end` tidak termasuk karakter akhir. Contoh: jika `text` adalah `Ada`, maka `text_to_annotate[start:end]` harus menghasilkan `Ada` persis.

In [ ]:
needed = ['text_to_annotate', 'annotations_json']
missing = [c for c in needed if c not in ann_df.columns]
if missing:
    raise ValueError(f'Kolom wajib tidak ada: {missing}. Gunakan file antrean dari Notebook 02.')

id_col = 'original_id' if 'original_id' in ann_df.columns else ('article_ref' if 'article_ref' in ann_df.columns else None)
if id_col is None:
    raise ValueError('Tidak ada original_id/article_ref untuk mencegah kebocoran antar split.')

records, errors = [], []
for row_idx, row in ann_df.iterrows():
    article_id = str(row[id_col]).strip()
    text_value = str(row['text_to_annotate'])
    raw_annotation = str(row['annotations_json']).strip()
    if not article_id or not text_value.strip():
        errors.append({'row': int(row_idx), 'article_id': article_id, 'error': 'ID artikel atau teks kosong'})
        continue
    if not raw_annotation:
        errors.append({'row': int(row_idx), 'article_id': article_id, 'error': 'annotations_json masih kosong; gunakan [] jika tidak ada entitas'})
        continue
    try:
        entities = json.loads(raw_annotation)
    except Exception as e:
        errors.append({'row': int(row_idx), 'article_id': article_id, 'error': f'JSON tidak valid: {e}'})
        continue
    if not isinstance(entities, list):
        errors.append({'row': int(row_idx), 'article_id': article_id, 'error': 'annotations_json harus berupa array JSON'})
        continue
    normalized = []
    spans = []
    for ent in entities:
        if not isinstance(ent, dict) or not all(k in ent for k in ('text','label','start','end')):
            errors.append({'row': int(row_idx), 'article_id': article_id, 'error': f'Format entitas salah: {ent}'})
            continue
        try:
            start, end = int(ent['start']), int(ent['end'])
        except Exception:
            errors.append({'row': int(row_idx), 'article_id': article_id, 'error': f'Offset bukan integer: {ent}'})
            continue
        label = str(ent['label']).strip().upper()
        span_text = str(ent['text'])
        if start < 0 or end <= start or end > len(text_value):
            errors.append({'row': int(row_idx), 'article_id': article_id, 'error': f'Offset di luar batas: {ent}'})
            continue
        if text_value[start:end] != span_text:
            errors.append({'row': int(row_idx), 'article_id': article_id, 'error': f'Teks offset tidak cocok: expected={text_value[start:end]!r}, annotated={span_text!r}'})
            continue
        if not label:
            errors.append({'row': int(row_idx), 'article_id': article_id, 'error': 'Label kosong'})
            continue
        spans.append((start, end, label, span_text))
        normalized.append({'text': span_text, 'label': label, 'start': start, 'end': end})
    spans.sort(key=lambda x: (x[0], x[1]))
    for prev, cur in zip(spans, spans[1:]):
        if cur[0] < prev[1]:
            errors.append({'row': int(row_idx), 'article_id': article_id, 'error': f'Span overlap: {prev[3]!r} vs {cur[3]!r}'})
    records.append({'article_id': article_id, 'text': text_value, 'entities': normalized, 'source_row': int(row_idx), 'annotator': str(row.get('annotator',''))})

if errors:
    error_df = pd.DataFrame(errors)
    display(error_df.head(50))
    error_df.to_csv(OUT/'annotation_validation_errors.csv', index=False)
    raise ValueError(f'Ditemukan {len(errors)} masalah anotasi. Perbaiki CSV lalu jalankan ulang. Lihat annotation_validation_errors.csv di output bila tersedia.')

# Satu artikel harus punya satu versi anotasi; jangan pilih salah satu secara diam-diam.
by_id = defaultdict(list)
for rec in records:
    by_id[rec['article_id']].append(rec)
conflicts = []
unique_records = []
for aid, items in by_id.items():
    signatures = {(x['text'], json.dumps(x['entities'], sort_keys=True, ensure_ascii=False)) for x in items}
    if len(signatures) > 1:
        conflicts.append({'article_id': aid, 'rows': [x['source_row'] for x in items], 'count_versions': len(signatures)})
    else:
        unique_records.append(items[0])
if conflicts:
    display(pd.DataFrame(conflicts))
    raise ValueError('Ada ID artikel duplikat dengan teks/anotasi berbeda. Selesaikan lewat diskusi kelompok sebelum training.')

records = unique_records
if len(records) < 10:
    raise ValueError(f'Hanya ada {len(records)} artikel unik valid. Dibutuhkan data lebih banyak untuk split dan tugas ini meminta minimal 10 artikel berlabel untuk evaluasi.')
all_labels = sorted({e['label'] for r in records for e in r['entities']})
if not all_labels:
    raise ValueError('Belum ada entitas berlabel di anotasi. Pastikan anotasi manual sudah diisi; artikel tanpa entitas boleh [], tetapi kumpulan data tidak boleh semua kosong.')
print('Artikel unik valid:', len(records))
print('Label ditemukan:', all_labels)
print('Jumlah span berlabel:', sum(len(r['entities']) for r in records))
display(pd.DataFrame([{'label': label, 'spans': sum(e['label']==label for r in records for e in r['entities'])} for label in all_labels]))

## 3. Bagi train/validation/test berdasarkan ID artikel

Pembagian mengikuti rekomendasi soal 70%/15%/15% dengan seed tetap. `GroupShuffleSplit` memastikan satu ID artikel tidak tersebar ke lebih dari satu split. Karena jumlah artikel bulat, persentase aktual dapat sedikit berbeda.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

texts = np.array([r['text'] for r in records], dtype=object)
groups = np.array([r['article_id'] for r in records], dtype=object)
idx_all = np.arange(len(records))

split1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(split1.split(texts, groups=groups))
temp_texts = texts[temp_idx]
temp_groups = groups[temp_idx]
split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_relative, test_relative = next(split2.split(temp_texts, groups=temp_groups))
val_idx = temp_idx[val_relative]
test_idx = temp_idx[test_relative]

splits = {
    'train': [records[i] for i in train_idx],
    'validation': [records[i] for i in val_idx],
    'test': [records[i] for i in test_idx],
}

def group_ids(part): return {r['article_id'] for r in part}
assert not (group_ids(splits['train']) & group_ids(splits['validation']))
assert not (group_ids(splits['train']) & group_ids(splits['test']))
assert not (group_ids(splits['validation']) & group_ids(splits['test']))

split_summary = []
for name, part in splits.items():
    split_summary.append({'split':name, 'documents':len(part), 'unique_article_ids':len(group_ids(part)), 'spans':sum(len(r['entities']) for r in part)})
    with open(OUT/f'{name}.jsonl','w',encoding='utf-8') as f:
        for r in part:
            f.write(json.dumps({'text':r['text'],'entities':r['entities'],'article_id':r['article_id']},ensure_ascii=False)+'\n')
display(pd.DataFrame(split_summary))
print('Catat seed:', SEED)

## 4. Siapkan spaCy dan baseline pretrained

Kita memakai model bahasa Inggris `en_core_web_sm` sebagai baseline. Baseline adalah model pretrained sebelum diadaptasi dengan anotasi kelompok. Skor akan dihitung dengan **exact-span matching**: teks, posisi awal/akhir, dan label harus cocok.

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch, fix_random_seed

try:
    baseline_nlp = spacy.load('en_core_web_sm')
except OSError:
    from spacy.cli import download
    download('en_core_web_sm')
    baseline_nlp = spacy.load('en_core_web_sm')

print('spaCy version:', spacy.__version__)
print('Baseline pipeline:', baseline_nlp.pipe_names)
print('Baseline NER labels:', baseline_nlp.get_pipe('ner').labels)

## 5. Fungsi evaluasi exact-span

Untuk setiap dokumen, sistem membandingkan tuple `(start, end, label)`. Precision = TP/(TP+FP), recall = TP/(TP+FN), dan F1 adalah rata-rata harmonik precision dan recall. Prediksi dengan label di luar skema anotasi kelompok diabaikan agar evaluasi sesuai kategori yang disepakati.

In [ ]:
def gold_tuples(record):
    return {(int(e['start']), int(e['end']), str(e['label'])) for e in record['entities']}

def predict_tuples(nlp, text, target_labels):
    doc = nlp(text)
    return {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents if ent.label_ in target_labels}

def metric_counts(gold_set, pred_set):
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return {'TP':tp,'FP':fp,'FN':fn,'precision':precision,'recall':recall,'f1':f1}

def evaluate_exact(nlp, data, target_labels):
    total_gold, total_pred = set(), set()
    # Gunakan indeks dokumen agar span yang sama di artikel berbeda tidak tercampur.
    tp = fp = fn = 0
    label_rows = []
    errors_list = []
    for i, rec in enumerate(data):
        g = gold_tuples(rec)
        p = predict_tuples(nlp, rec['text'], target_labels)
        correct = g & p
        missed = g - p
        extra = p - g
        tp += len(correct); fn += len(missed); fp += len(extra)
        for start,end,label in missed:
            errors_list.append({'article_id':rec['article_id'],'kind':'false_negative','label':label,'span':rec['text'][start:end],'start':start,'end':end,'text_excerpt':rec['text'][:500]})
        for start,end,label in extra:
            errors_list.append({'article_id':rec['article_id'],'kind':'false_positive','label':label,'span':rec['text'][start:end],'start':start,'end':end,'text_excerpt':rec['text'][:500]})
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    overall = {'TP':tp,'FP':fp,'FN':fn,'precision':precision,'recall':recall,'f1':f1}
    for label in sorted(target_labels):
        ltp = lfp = lfn = 0
        for rec in data:
            g = {x for x in gold_tuples(rec) if x[2] == label}
            p = {x for x in predict_tuples(nlp, rec['text'], target_labels) if x[2] == label}
            ltp += len(g & p); lfp += len(p-g); lfn += len(g-p)
        lp = ltp/(ltp+lfp) if ltp+lfp else 0.0
        lr = ltp/(ltp+lfn) if ltp+lfn else 0.0
        lf = 2*lp*lr/(lp+lr) if lp+lr else 0.0
        label_rows.append({'label':label,'TP':ltp,'FP':lfp,'FN':lfn,'precision':lp,'recall':lr,'f1':lf})
    return overall, pd.DataFrame(label_rows), pd.DataFrame(errors_list)

TARGET_LABELS = set(all_labels)
baseline_val, baseline_val_per_label, baseline_val_errors = evaluate_exact(baseline_nlp, splits['validation'], TARGET_LABELS)
print('Baseline validation (exact span):')
display(pd.DataFrame([baseline_val]))
display(baseline_val_per_label)

## 6. Adaptasi model pretrained pada data training

Sel ini melatih komponen NER dari model pretrained. Kita memakai validation F1 untuk memilih checkpoint terbaik (early stopping). Test set tidak digunakan pada pemilihan checkpoint. Hasil dapat bervariasi menurut data dan runtime, jadi catat waktu serta parameter aktual.

In [ ]:
fix_random_seed(SEED)
trained_nlp = spacy.load('en_core_web_sm')
ner = trained_nlp.get_pipe('ner')
for label in TARGET_LABELS:
    ner.add_label(label)

# Validasi alignment terhadap tokenizer: setiap span harus dimulai/berakhir pada batas token.
def to_spacy_example(nlp, record):
    entities = []
    doc = nlp.make_doc(record['text'])
    for ent in record['entities']:
        span = doc.char_span(int(ent['start']), int(ent['end']), label=ent['label'], alignment_mode='strict')
        if span is None:
            raise ValueError(f"Offset tidak sejajar token: article={record['article_id']}, entitas={ent}")
        entities.append((int(ent['start']), int(ent['end']), str(ent['label'])))
    return Example.from_dict(doc, {'entities': entities})

train_examples = [to_spacy_example(trained_nlp, r) for r in splits['train']]
# Validasi alignment pada validation juga, tetapi tidak dipakai update bobot.
val_examples = [to_spacy_example(trained_nlp, r) for r in splits['validation']]
print('Train examples:', len(train_examples), '| Validation examples:', len(val_examples))

optimizer = trained_nlp.resume_training()
best_val_f1 = -1.0
best_epoch = 0
wait = 0
history = []
best_model_dir = OUT/'best_model'

train_components = [name for name in trained_nlp.pipe_names if name in {'tok2vec', 'transformer', 'ner'}]
with trained_nlp.select_pipes(enable=train_components):
    for epoch in range(1, MAX_EPOCHS+1):
        random.shuffle(train_examples)
        losses = {}
        batches = minibatch(train_examples, size=8)
        for batch in batches:
            trained_nlp.update(batch, sgd=optimizer, drop=0.20, losses=losses)
        val_overall, val_by_label, _ = evaluate_exact(trained_nlp, splits['validation'], TARGET_LABELS)
        row = {'epoch':epoch,'ner_loss':float(losses.get('ner',0.0)), 'validation_precision':val_overall['precision'],'validation_recall':val_overall['recall'],'validation_f1':val_overall['f1']}
        history.append(row)
        print(row)
        if val_overall['f1'] > best_val_f1:
            best_val_f1 = val_overall['f1']
            best_epoch = epoch
            wait = 0
            if best_model_dir.exists(): shutil.rmtree(best_model_dir)
            trained_nlp.to_disk(best_model_dir)
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'Early stopping pada epoch {epoch}; checkpoint terbaik epoch {best_epoch}.')
                break

history_df = pd.DataFrame(history)
history_df.to_csv(OUT/'training_history.csv', index=False)
display(history_df)
print('Epoch terbaik:', best_epoch, '| Validation F1 terbaik:', round(best_val_f1,4))

## 7. Muat checkpoint terbaik dan evaluasi final

Setelah pemilihan model selesai dengan validation, kita muat checkpoint terbaik. Bagian berikut adalah evaluasi final pada test set yang ditahan. Jangan gunakan hasil test ini untuk mengubah label, preprocessing, atau parameter; jika ada perbaikan setelah melihat test, siapkan test baru untuk evaluasi yang tidak bias.

In [ ]:
best_nlp = spacy.load(best_model_dir)
val_final, val_final_by_label, val_final_errors = evaluate_exact(best_nlp, splits['validation'], TARGET_LABELS)
test_baseline, test_baseline_by_label, test_baseline_errors = evaluate_exact(baseline_nlp, splits['test'], TARGET_LABELS)
test_trained, test_trained_by_label, test_trained_errors = evaluate_exact(best_nlp, splits['test'], TARGET_LABELS)

comparison = pd.DataFrame([
    {'model':'Pretrained baseline','split':'validation','precision':baseline_val['precision'],'recall':baseline_val['recall'],'f1':baseline_val['f1']},
    {'model':'Adapted model (best checkpoint)','split':'validation','precision':val_final['precision'],'recall':val_final['recall'],'f1':val_final['f1']},
    {'model':'Pretrained baseline','split':'test','precision':test_baseline['precision'],'recall':test_baseline['recall'],'f1':test_baseline['f1']},
    {'model':'Adapted model (final)','split':'test','precision':test_trained['precision'],'recall':test_trained['recall'],'f1':test_trained['f1']},
])
display(comparison.style.format({'precision':'{:.3f}','recall':'{:.3f}','f1':'{:.3f}'}))
comparison.to_csv(OUT/'baseline_vs_trained_metrics.csv', index=False)
val_final_by_label.to_csv(OUT/'validation_metrics_per_label.csv', index=False)
test_trained_by_label.to_csv(OUT/'test_metrics_per_label.csv', index=False)
test_trained_errors.to_csv(OUT/'test_errors.csv', index=False)

print('Per-label test metrics:')
display(test_trained_by_label)
print('Contoh kesalahan model final pada test set:')
display(test_trained_errors.head(10))
if len(test_trained_errors) < 3:
    print('Catatan: ditemukan kurang dari 3 error otomatis. Laporan tetap harus membahas minimal tiga contoh kesalahan/kasus sulit; tambahkan analisis kualitatif yang transparan, jangan mengarang error.')

## 8. Simpan model dan laporan

Arsip berisi model terbaik dan metrik. Jaga agar data artikel penuh tidak ikut masuk ke repository publik; artefak model juga bisa berukuran besar, jadi unduh dan simpan di Drive kelompok atau tempat yang disetujui dosen.

In [ ]:
# Metadata reprodusibilitas
meta = {
    'base_model':'en_core_web_sm',
    'spaCy_version':spacy.__version__,
    'seed':SEED,
    'max_epochs':MAX_EPOCHS,
    'patience':PATIENCE,
    'best_epoch':int(best_epoch),
    'target_labels':sorted(TARGET_LABELS),
    'split_counts':{k:len(v) for k,v in splits.items()},
    'training_documents':len(splits['train']),
    'validation_documents':len(splits['validation']),
    'test_documents':len(splits['test']),
    'annotation_source':'human-reviewed CSV files supplied by the three group members',
    'matching':'exact-span (start, end, label)'
}
(OUT/'model_metadata.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
# Salin model agar folder output mudah diunduh.
export_model_dir = OUT/'model'
if export_model_dir.exists(): shutil.rmtree(export_model_dir)
shutil.copytree(best_model_dir, export_model_dir)
archive_path = shutil.make_archive('/content/NER_Trained_Model_and_Report','zip',OUT)
print('Model + hasil evaluasi dibuat:', archive_path)
print('File laporan:', sorted(p.name for p in OUT.iterdir()))
files.download(archive_path)

## 9. Interpretasi dan item laporan

Laporan harus menyertakan:

- jumlah artikel dan span pada setiap split;
- seed, versi spaCy, nama model baseline, epoch terbaik, waktu training (catat waktu aktual jika diperlukan), dan distribusi label;
- precision, recall, F1 exact-span overall pada validation dan test;
- metrik per label;
- baseline vs adapted model;
- minimal tiga contoh false positive/false negative atau kasus sulit beserta penjelasan;
- konfirmasi bahwa test set tidak dipakai untuk tuning;
- keterbatasan: jumlah anotasi, bahasa yang didukung, ketidakseimbangan label, panjang artikel, dan konsistensi anotasi.

**Jangan menyalin metrik contoh dari internet atau diagram. Gunakan angka hasil eksekusi notebook kalian.**